# The Boltzmann Distribution

The point of this notebook is just to run quick calculations whenever I'm working through my
physics textbook and there is some beefy integral/other numerical calculation I need to take care of
and I don't want to have to create a new .py file just for one calculation

Calculates the speed and probability of copper electron escaping copper atom given temperature

In [ ]:
import math
from scipy import constants as const
import matplotlib.pyplot as plt
import numpy as np
from scipy.integrate import quad

k = const.k #Joules / Kelvin
T = 1000 #Kelvin
m_e = const.m_e #Kg
E = 5 #eV

E_j = (1.6e-19) * E #Energy in Joules
v_min = math.sqrt((2 * E_j)/m_e) # Minimum speed electron needs to escape copper atom

print(f"Minimum speed an electron needs to escape copper atom is roughly {v_min:.3e} m/s")

def Maxwell_Speed_Distribution(v, m, T):
    return 4*np.pi * (m/(2*np.pi*k*T))**(3/2) * v**2 * np.exp(-m*v**2/(2*k*T))

P = quad(Maxwell_Speed_Distribution, v_min, np.inf, args=(m_e, T))

print(f"Probability an electron having enough energy to escape is {P[0]:0.3e}")


Maxwell's speed distribution for Air Molecules

In [ ]:
v_nm = 517 #Avergae velocity of nitrogen molecule in a typical room in m/s
m_nm = 4.65e-26 #mass of nitrogen molecule in Kg
T_room = 300 #Room temperature in Kelvin

print(f"Probability nitrogen moleculeis within 50 m/s of is {quad(Maxwell_Speed_Distribution, v_nm - 50, v_nm + 50, args=(m_nm, T_room))[0]:0.3f}")
print(f"Probability a given molecule is more than twice the average speed is {quad(Maxwell_Speed_Distribution, 2 * v_nm, np.inf, args=(m_nm, T_room))[0]:0.3f}")

Calculates <E> given discrete energy levels. Enter how much each level increases by and how many energy levels you want

In [ ]:
def E_av(levels, Delta_E):
    Z = 0.0
    for E in range(levels):
        Z += math.exp(-E * Delta_E)
    E_av = 0.0
    for E in range(levels):
        E_av += ((E * Delta_E) / Z) * math.exp(-E * Delta_E)
    return E_av
    
print(f"Average energy with 3 distinct energy levels differing by 3kT is {E_av(3, 3):0.3f}kT")
print(f"Average energy with 3 distinct energy levels differing by 6kT is {E_av(3, 6):0.3f}kT")
print(f"Average energy with 60 distinct energy levels differing by (1/10)kT is {E_av(60, 0.1):0.3f}kT")

# Quantum Statistics

When particles are completely indistinguishable, such as bosons and fermions, their statistical behaviors change due to quantum statistics. Fermions and bosons follow a distribution different from the Boltzmann distribution.

In [ ]:
# Calculates the occupancy of fermions for different values of E that are near the chemical potential (u)
import math
from scipy import constants as const
import matplotlib.pyplot as plt
import numpy as np
from scipy.integrate import quad

k = const.k #Joules / Kelvin
k_ev = k / const.e # Boltzman Constant in eV

def Fermi_Dirac_eV(Diff_E, T):
    return 1 / (np.exp(Diff_E / (k_ev * T)) + 1)

T_room = 300 #Kelvin

print(f"For E 0.01 eV less than u, n = {Fermi_Dirac_eV(-0.01, T_room):0.3f}")
print(f"For E = u, n = {Fermi_Dirac_eV(0, T_room):0.3f}")
print(f"For E 0.01 eV more than u, n = {Fermi_Dirac_eV(0.01, T_room):0.3f}")

For photons passing through a rouhgly 1D container (such as a fiber obtic cable), if they are in thermal equilibrium with the gas surrounding them, the possible energies for particles in this state are E_n = (pi * hbar * c * n) / L, where n is a non-negative integer. The total number of photons in this box is the infinite series of the Bose-Einstein distribution where its energies are E_n with n from 0 to infinity. This program numerically approximates that infinite summation by adding terms until additional terms are negligible.

In [ ]:
def occupation(n):
    En = np.pi * const.hbar * const.c * n / L
    x = En / (const.k * T_room)
    return 1.0 / np.expm1(x)   # more stable than exp(x) - 1

total = 0.0
n = 1

while True:
    term = occupation(n)
    total += term

    # stop when additional terms are negligible
    if term < 1e-12:
        break

    n += 1

print("n_max used =", n)
print(f"Total number of photons = {total:0.3e}")

Several Plots of the Fermi-Dirac distribution to view the properties of the distribution

In [ ]:
# Energy range (both negative and positive)
E_range = np.linspace(-0.5, 0.5, 1000)  # eV

# Temperatures to plot
temperatures = [50, 300, 1000, 3000, 5000]  # Kelvin


# Plot
plt.figure(figsize=(8, 6))

for T in temperatures:
    f = Fermi_Dirac_eV(E_range, T)
    plt.plot(E_range, f, label=f"T = {T} K")

# Formatting
plt.axvline(linestyle='--', label=r'$\mu = 0$')
plt.xlabel("Energy (eV)")
plt.ylabel("Occupation Probability")
plt.title("Fermi–Dirac Distribution at Various Temperatures")
plt.legend()
plt.grid(True)

plt.show()

# Blackbody Radiation

Plot showing the difference between the classically predicted Rayleigh-Jeans spectrum and Plank's spectrum

In [ ]:
import math
from scipy import constants as const
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
from scipy.integrate import quad

T = 300 #Kelvin
k = const.k #Boltzmann constant
c = const.c
h = const.h

def Classical_Spectrum(v):
    return (8 * np.pi * k * T * (v ** 2)) / (c ** 3)

def Planck_Spectrum(v):
    return ((8 * np.pi * h * (v ** 3)) / (c ** 3)) / (np.expm1((h * v) / (k * T)))

freq_planck_THz = np.linspace(1, 60, 200)
freq_classic_THz = np.linspace(1, 8.8, 200)

y_vals_classic = Classical_Spectrum(freq_classic_THz * 1e12)
y_vals_planck = Planck_Spectrum(freq_planck_THz * 1e12)

plt.figure(figsize = (8, 6))
plt.plot(freq_classic_THz, y_vals_classic, label='Classical Frequency Spectrum', color='b')
plt.plot(freq_planck_THz, y_vals_planck, label='Plancks Frequency Spectrum', color='k')
plt.xlabel("Frequency (THz)")
plt.ylabel("Spectral Energy Density per Hert (J s / m^3)")
plt.title("Energy Density vs Frequency (Ultraviolet Catastrophe)")
plt.legend()
plt.xticks(np.arange(0, 61, 5))
plt.xlim(0, 60)
plt.ylim(0, 3e-19)
plt.tick_params(axis='both', direction='in')
plt.show()


Here are some changes to test

In [ ]:
#testing more code changes

In [ ]:
#idk why jupyter sucks